In [0]:
# NB_gold_validation — Gold Layer Validation Suite
# Purpose: Validate schema, grain, nullability, duplication, reconciliation, and rate calculations
# Targets: gold.patient_360, gold.hospital_daily_summary, gold.lab_test_trends

from pyspark.sql import functions as F
import uuid
from datetime import datetime
import json

CATALOG = "caresync_databricks_workspace"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

RUN_ID = str(uuid.uuid4())
RUN_START = datetime.utcnow()
TEST_RESULTS = []

print("=" * 60)
print("NB_gold_validation — Gold Layer Validation")
print("=" * 60)
print(f"  Run ID             : {RUN_ID}")
print(f"  Run Start (UTC)    : {RUN_START}")
print("=" * 60)


def record_test(test_name, passed, details):
    status = "PASS" if passed else "FAIL"
    TEST_RESULTS.append({
        "test_name": test_name,
        "status": status,
        "details": details
    })
    print(f"[{status}] {test_name}: {details}")


def assert_and_record(test_name, condition, success_details, failure_details):
    if condition:
        record_test(test_name, True, success_details)
    else:
        record_test(test_name, False, failure_details)


In [0]:
print("=" * 60)
print("STEP 1: Load Source and Target Tables")
print("=" * 60)

patients_src = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.patients")
lab_results_src = spark.table(f"{CATALOG}.{SOURCE_SCHEMA}.lab_results")
patient_360 = spark.table(f"{CATALOG}.{TARGET_SCHEMA}.patient_360")
hospital_daily = spark.table(f"{CATALOG}.{TARGET_SCHEMA}.hospital_daily_summary")
lab_test_trends = spark.table(f"{CATALOG}.{TARGET_SCHEMA}.lab_test_trends")

print(f"  patient_360             : {patient_360.count():,} rows")
print(f"  hospital_daily_summary  : {hospital_daily.count():,} rows")
print(f"  lab_test_trends         : {lab_test_trends.count():,} rows")
print("=" * 60)

In [0]:
print("=" * 60)
print("STEP 2: Validate patient_360")
print("=" * 60)

# Schema check
expected_patient_360_cols = [
    "patient_id", "first_name", "last_name", "date_of_birth", "age", "gender",
    "city", "state", "hospital_id", "hospital_name", "insurance_provider_id",
    "insurance_provider_name", "total_tests", "abnormal_test_count",
    "critical_test_count", "last_test_date"
]
actual_cols = patient_360.columns
assert_and_record(
    "patient_360 schema",
    actual_cols == expected_patient_360_cols,
    f"Columns match STTM ({len(actual_cols)} columns)",
    f"Schema mismatch. Expected {expected_patient_360_cols}, got {actual_cols}"
)

# Row count reconciliation: patient_360 should equal patients
src_patients_count = patients_src.count()
tgt_patients_count = patient_360.count()
assert_and_record(
    "patient_360 row count reconciliation",
    tgt_patients_count == src_patients_count,
    f"Target rows {tgt_patients_count:,} = source patients {src_patients_count:,}",
    f"Target rows {tgt_patients_count:,} != source patients {src_patients_count:,}"
)

# Uniqueness at grain
patient_dupes = patient_360.groupBy("patient_id").count().filter("count > 1").count()
assert_and_record(
    "patient_360 grain uniqueness",
    patient_dupes == 0,
    "No duplicate patient_id values",
    f"Found {patient_dupes} duplicate patient_id values"
)

# Nullability checks
null_patient_id = patient_360.filter("patient_id IS NULL").count()
assert_and_record(
    "patient_360 primary key non-null",
    null_patient_id == 0,
    "No null patient_id values",
    f"Found {null_patient_id} null patient_id values"
)

# Counts sanity
bad_counts = patient_360.filter(
    (F.col("total_tests") < 0) |
    (F.col("abnormal_test_count") < 0) |
    (F.col("critical_test_count") < 0) |
    (F.col("abnormal_test_count") > F.col("total_tests")) |
    (F.col("critical_test_count") > F.col("total_tests"))
).count()
assert_and_record(
    "patient_360 count sanity",
    bad_counts == 0,
    "All counts are non-negative and bounded by total_tests",
    f"Found {bad_counts} invalid count rows"
)

# Patients with no labs should exist and have zero counts
no_lab_rows = patient_360.filter(
    (F.col("total_tests") == 0) &
    (F.col("abnormal_test_count") == 0) &
    (F.col("critical_test_count") == 0) &
    (F.col("last_test_date").isNull())
).count()
assert_and_record(
    "patient_360 no-lab handling",
    no_lab_rows >= 3,
    f"Found {no_lab_rows} rows with zero-count/no-last_test_date handling",
    f"Expected at least 3 no-lab rows, found {no_lab_rows}"
)

In [0]:
print("=" * 60)
print("STEP 3: Validate hospital_daily_summary")
print("=" * 60)

expected_hospital_daily_cols = [
    "hospital_id", "hospital_name", "summary_date", "day_name", "is_weekend",
    "total_tests", "unique_patients_tested", "active_doctor_count",
    "abnormal_test_count", "critical_test_count", "abnormal_rate_pct",
    "critical_rate_pct", "prev_day_total_tests", "test_volume_change_pct"
]
actual_cols = hospital_daily.columns
assert_and_record(
    "hospital_daily_summary schema",
    actual_cols == expected_hospital_daily_cols,
    f"Columns match STTM ({len(actual_cols)} columns)",
    f"Schema mismatch. Expected {expected_hospital_daily_cols}, got {actual_cols}"
)

# Grain uniqueness
hospital_daily_dupes = hospital_daily.groupBy("hospital_id", "summary_date").count().filter("count > 1").count()
assert_and_record(
    "hospital_daily_summary grain uniqueness",
    hospital_daily_dupes == 0,
    "No duplicate (hospital_id, summary_date) rows",
    f"Found {hospital_daily_dupes} duplicate grain rows"
)

# Nullability of grain keys
null_grain = hospital_daily.filter("hospital_id IS NULL OR summary_date IS NULL").count()
assert_and_record(
    "hospital_daily_summary grain non-null",
    null_grain == 0,
    "No null grain keys",
    f"Found {null_grain} rows with null hospital_id or summary_date"
)

# Reconciliation: sum(total_tests) == source lab_results
src_lab_count = lab_results_src.count()
hospital_daily_total_tests = hospital_daily.agg(F.sum("total_tests").alias("s")).collect()[0]["s"]
assert_and_record(
    "hospital_daily_summary total test reconciliation",
    hospital_daily_total_tests == src_lab_count,
    f"SUM(total_tests) {hospital_daily_total_tests:,} = source lab_results {src_lab_count:,}",
    f"SUM(total_tests) {hospital_daily_total_tests:,} != source lab_results {src_lab_count:,}"
)

# Rate range checks
bad_rates = hospital_daily.filter(
    (F.col("abnormal_rate_pct") < 0) | (F.col("abnormal_rate_pct") > 100) |
    (F.col("critical_rate_pct") < 0) | (F.col("critical_rate_pct") > 100)
).count()
assert_and_record(
    "hospital_daily_summary rate bounds",
    bad_rates == 0,
    "All rates are between 0 and 100",
    f"Found {bad_rates} rows with invalid rate bounds"
)

# LAG sanity: first row per hospital should have null prev_day_total_tests
window_check = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM (
  SELECT hospital_id, summary_date, prev_day_total_tests,
         ROW_NUMBER() OVER (PARTITION BY hospital_id ORDER BY summary_date) AS rn
  FROM {CATALOG}.{TARGET_SCHEMA}.hospital_daily_summary
) x
WHERE rn = 1 AND prev_day_total_tests IS NOT NULL
""").collect()[0]["cnt"]
assert_and_record(
    "hospital_daily_summary lag first-row null",
    window_check == 0,
    "First row per hospital has NULL prev_day_total_tests",
    f"Found {window_check} first rows with non-NULL prev_day_total_tests"
)

In [0]:
print("=" * 60)
print("STEP 4: Validate lab_test_trends")
print("=" * 60)

expected_lab_test_trends_cols = [
    "test_name", "year", "month", "month_name", "quarter_name",
    "total_tests", "abnormal_test_count", "critical_test_count",
    "abnormal_rate_pct", "critical_rate_pct", "prev_month_abnormal_rate_pct",
    "abnormal_rate_change_pct"
]
actual_cols = lab_test_trends.columns
assert_and_record(
    "lab_test_trends schema",
    actual_cols == expected_lab_test_trends_cols,
    f"Columns match STTM ({len(actual_cols)} columns)",
    f"Schema mismatch. Expected {expected_lab_test_trends_cols}, got {actual_cols}"
)

# Grain uniqueness
lab_trend_dupes = lab_test_trends.groupBy("test_name", "year", "month").count().filter("count > 1").count()
assert_and_record(
    "lab_test_trends grain uniqueness",
    lab_trend_dupes == 0,
    "No duplicate (test_name, year, month) rows",
    f"Found {lab_trend_dupes} duplicate grain rows"
)

# Nullability of grain keys
null_grain = lab_test_trends.filter("test_name IS NULL OR year IS NULL OR month IS NULL").count()
assert_and_record(
    "lab_test_trends grain non-null",
    null_grain == 0,
    "No null grain keys",
    f"Found {null_grain} rows with null test_name/year/month"
)

# Reconciliation: sum(total_tests) == source lab_results
lab_trends_total_tests = lab_test_trends.agg(F.sum("total_tests").alias("s")).collect()[0]["s"]
assert_and_record(
    "lab_test_trends total test reconciliation",
    lab_trends_total_tests == src_lab_count,
    f"SUM(total_tests) {lab_trends_total_tests:,} = source lab_results {src_lab_count:,}",
    f"SUM(total_tests) {lab_trends_total_tests:,} != source lab_results {src_lab_count:,}"
)

# Rate range checks
bad_rates = lab_test_trends.filter(
    (F.col("abnormal_rate_pct") < 0) | (F.col("abnormal_rate_pct") > 100) |
    (F.col("critical_rate_pct") < 0) | (F.col("critical_rate_pct") > 100)
).count()
assert_and_record(
    "lab_test_trends rate bounds",
    bad_rates == 0,
    "All rates are between 0 and 100",
    f"Found {bad_rates} rows with invalid rate bounds"
)

# quarter_name format check
bad_quarter_names = lab_test_trends.filter(~F.col("quarter_name").rlike("^Q[1-4]$")) .count()
assert_and_record(
    "lab_test_trends quarter_name format",
    bad_quarter_names == 0,
    "All quarter_name values match Q1-Q4",
    f"Found {bad_quarter_names} rows with invalid quarter_name format"
)

# LAG sanity: first row per test_name should have null prev_month_abnormal_rate_pct
window_check = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM (
  SELECT test_name, year, month, prev_month_abnormal_rate_pct,
         ROW_NUMBER() OVER (PARTITION BY test_name ORDER BY year, month) AS rn
  FROM {CATALOG}.{TARGET_SCHEMA}.lab_test_trends
) x
WHERE rn = 1 AND prev_month_abnormal_rate_pct IS NOT NULL
""").collect()[0]["cnt"]
assert_and_record(
    "lab_test_trends lag first-row null",
    window_check == 0,
    "First row per test_name has NULL prev_month_abnormal_rate_pct",
    f"Found {window_check} first rows with non-NULL prev_month_abnormal_rate_pct"
)

In [0]:
print("=" * 60)
print("STEP 5: Validation Summary")
print("=" * 60)

passed = sum(1 for x in TEST_RESULTS if x["status"] == "PASS")
failed = sum(1 for x in TEST_RESULTS if x["status"] == "FAIL")

summary = {
    "status": "SUCCESS" if failed == 0 else "FAILED",
    "run_id": RUN_ID,
    "tests_passed": passed,
    "tests_failed": failed,
    "run_start_utc": str(RUN_START),
    "run_end_utc": str(datetime.utcnow()),
    "results": TEST_RESULTS
}

print(f"  Tests passed : {passed}")
print(f"  Tests failed : {failed}")
print("=" * 60)

if failed > 0:
    raise AssertionError(json.dumps(summary, indent=2))
else:
    dbutils.notebook.exit(json.dumps(summary))